# Textile defect detection - Colab training notebook

Trains the ResNet-18 baseline on a GPU **if Colab gives you one** (free GPU availability, session length and memory vary - check Colab's current documentation). Run the cells top to bottom.

**Honesty rules:** this notebook only produces results you actually run. A model trained on the *synthetic* demo data proves the software works - it says nothing about real textile defects. Real results need a real, licensed dataset that you have checked (see `docs/DATASETS.md`).

Colab's `/content` storage is temporary. Cell 7 copies results to Google Drive / downloads them.

## 1. Check the runtime (GPU optional)

In [ ]:
import sys, platform
import torch
print("Python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Training still works but is slow. Runtime > Change runtime type > GPU.")

## 2. Get the project code
**Option A (works for a private/unpublished project):** upload `textile-defect-detection.zip` in the Colab file panel, then run the cell below.
**Option B:** if you already published the repository publicly, set `REPO_URL` and use `git clone` instead.

In [ ]:
import os, zipfile, pathlib
ZIP = "/content/textile-defect-detection.zip"
if os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/content")
else:
    # Option B (public repo only):
    # !git clone https://github.com/vipintk2807/textile-defect-detection.git /content/textile-defect-detection
    raise FileNotFoundError("Upload textile-defect-detection.zip to /content first (or use the git clone option).")
%cd /content/textile-defect-detection

## 3. Install dependencies
Colab already ships PyTorch/torchvision; we only add the small remaining packages and install the project **without** re-installing torch.

In [ ]:
!pip install -q pyyaml scikit-learn pillow
!pip install -q -e . --no-deps
import textile_defect_detection; print("project version", textile_defect_detection.__version__)

## 4. Choose the dataset
* **`USE_SYNTHETIC = True`** - generates the labelled SYNTHETIC demo dataset (software smoke test only).
* **`USE_SYNTHETIC = False`** - use your real dataset. Upload it (or mount Drive) so that it has `<class>/<image>` folders under `DATA_ROOT`, add a `dataset_info.json` with its real source/licence (copy `data/README.md` guidance), and check that its licence permits uploading it to Colab/Drive.

In [ ]:
USE_SYNTHETIC = True
DATA_ROOT = "/content/data/synthetic_demo" if USE_SYNTHETIC else "/content/drive/MyDrive/YOUR_DATASET_FOLDER"
PROCESSED = "/content/data/processed"
GROUP_REGEX = r"_(g\d+)_" if USE_SYNTHETIC else ""   # real data: regex that extracts a specimen/session id, or leave empty

In [ ]:
if USE_SYNTHETIC:
    !python scripts/make_synthetic_fixture.py --out {DATA_ROOT} --images-per-class 40
group_arg = f'--group-regex "{GROUP_REGEX}"' if GROUP_REGEX else ""
!python scripts/prepare_dataset.py --source-root {DATA_ROOT} --output-dir {PROCESSED} {group_arg}
!python scripts/validate_dataset.py --manifest {PROCESSED}/samples.csv --images-root {DATA_ROOT}

Stop here if validation reports **errors** (leakage, missing files, invalid labels). Warnings need your review but do not block training.

## 5. Train
Settings are in a YAML file. The cell writes a Colab-specific copy of the config (GPU if available). For the synthetic demo it trains from scratch with tiny images; for real data it uses ImageNet-pretrained ResNet-18 (downloaded once; needs internet).

In [ ]:
import yaml, pathlib
cfg = {
  "dataset": {"manifest_path": f"{PROCESSED}/samples.csv", "images_root": DATA_ROOT},
  "model": {"architecture": "resnet18", "pretrained": not USE_SYNTHETIC, "freeze_backbone": USE_SYNTHETIC is False},
  "preprocessing": ({"resize_size": 72, "crop_size": 64, "resize_mode": "squash"} if USE_SYNTHETIC else {}),
  "augmentation": {"enabled": True},
  "training": {"seed": 42, "batch_size": 32 if not USE_SYNTHETIC else 8, "max_epochs": 20 if not USE_SYNTHETIC else 5,
               "early_stopping_patience": 5, "device": "auto", "num_workers": 2},
  "output": {"directory": "/content/outputs/experiments", "experiment_name": "colab-run"},
}
pathlib.Path("configs").mkdir(exist_ok=True)
with open("configs/colab_training.yaml", "w") as f: yaml.safe_dump(cfg, f)
!python scripts/train.py --config configs/colab_training.yaml

## 6. Evaluate on the held-out test split - **once**
Only run this after you have finished every decision (epochs, thresholds, preprocessing). Re-running and re-tuning against the test split makes it a validation set; the report counts previous evaluations of the split and warns you.

In [ ]:
import glob, os
exp = sorted(glob.glob("/content/outputs/experiments/*colab-run"))[-1]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Experiment:", exp, "| evaluating on", DEVICE)
!python scripts/evaluate.py --model-dir {exp}/model --manifest {PROCESSED}/samples.csv --images-root {DATA_ROOT} --split test --device {DEVICE}


## 7. Save results outside Colab's temporary storage
Copies the experiment folder (config, logs, class mapping, manifests, model, evaluation) into a ZIP you download, and optionally to Google Drive.

In [ ]:
import shutil
archive = shutil.make_archive("/content/experiment_results", "zip", exp)
print("Created", archive)
try:
    from google.colab import files, drive
    files.download(archive)                    # browser download
    # Optional Drive copy:
    # drive.mount("/content/drive"); shutil.copytree(exp, "/content/drive/MyDrive/textile_experiments/" + os.path.basename(exp))
except ImportError:
    print("Not running in Colab; the ZIP is at", archive)

## 8. Use the model on your Windows PC
Unzip the download. Copy `model/model_weights.pth` and `model/model_metadata.json` into the project's `models` folder (or run `python scripts/export_model.py --model-dir <unzipped>/model` locally), then `python -m streamlit run app.py`.